# WALMART LMI ANALYSIS
### Healthcare + Manufacturing + Semiconductors
### Short-term and Long-term Industry/Occupation Projections

In [69]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


# =====================================================================
# FILE PATHS
# =====================================================================

BASE = Path(
    r"C:\Users\301533\Documents\Walmart_Analysis_Output\SOCS& NAICS"
)

SECTOR_FILES = {
    "Healthcare": (
        BASE
        / "Walmart industry with occupations Healthcare 09.2026 (1).xlsx"
    ),
    "Semiconductors": (
        BASE
        / "Walmart Matched Occupations (semiconductors- aca definition) (1).xlsx"
    ),
    "Manufacturing": (
        BASE
        / "Walmart industry with occupations Manufacturing 09.2026.xlsx"
    ),
}

SHORT_TERM_OCCUPATION_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\occ_short_term (6).xlsx"
)

LONG_TERM_OCCUPATION_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\ltop-04-2024to2034 (7).xlsx"
)

SHORT_TERM_INDUSTRY_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\stip-04-2025to2027 (2).xlsx"
)

LONG_TERM_INDUSTRY_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\ltip-04-2024to2034 (3).xlsx"
)

OUTPUT_FOLDER = Path(
    r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)

OUTPUT_FILE = (
    OUTPUT_FOLDER
    / "Walmart_Sector_Analysis.xlsx"
)


# =====================================================================
# FORMATTING COLORS
# =====================================================================

TEAL = "369992"
GRAY = "D8D8D8"
LIGHT_GRAY = "F2F2F2"
WHITE = "FFFFFF"
BLACK = "000000"


# =====================================================================
# CLEANING FUNCTIONS
# =====================================================================

def clean_text(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def normalized(value):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        clean_text(value).lower(),
    ).strip()


def make_unique_columns(columns):
    """
    Make duplicate Excel headings unique.

    Example:
        Percent
        Percent_2
        Percent_3
    """

    counts = {}
    unique_columns = []

    for column in columns:

        column = clean_text(column)

        if column not in counts:
            counts[column] = 1
            unique_columns.append(column)

        else:
            counts[column] += 1
            unique_columns.append(
                f"{column}_{counts[column]}"
            )

    return unique_columns


def normalize_soc(value):
    """Convert SOC codes into ##-#### format."""

    if pd.isna(value):
        return None

    text = clean_text(value)

    match = re.search(
        r"(?<!\d)(\d{2})[- ]?(\d{4})(?!\d)",
        text,
    )

    if match:
        return (
            f"{match.group(1)}-"
            f"{match.group(2)}"
        )

    digits = re.sub(
        r"\D",
        "",
        text.split(".")[0],
    )

    if len(digits) == 6:
        return (
            f"{digits[:2]}-"
            f"{digits[2:]}"
        )

    return None


def normalize_naics(value):
    """Return a three-digit NAICS code."""

    if pd.isna(value):
        return None

    text = clean_text(value)

    # Exclude aggregate ranges such as 31-33.
    if "-" in text:
        return None

    digits = re.sub(
        r"\D",
        "",
        text,
    )

    if len(digits) >= 3:
        return digits[:3]

    return None


def to_number(data):
    """
    Convert currency, percentage, comma, and parenthetical values
    into numbers.
    """

    if isinstance(data, pd.DataFrame):
        data = data.iloc[:, 0]

    text = data.astype(str).str.strip()

    percent_mask = text.str.contains(
        "%",
        regex=False,
        na=False,
    )

    cleaned = (
        text
        .str.replace(
            r"[$,+]",
            "",
            regex=True,
        )
        .str.replace(
            "%",
            "",
            regex=False,
        )
        .str.replace(
            r"^\((.*)\)$",
            r"-\1",
            regex=True,
        )
        .replace(
            {
                "": np.nan,
                "nan": np.nan,
                "None": np.nan,
            }
        )
    )

    values = pd.to_numeric(
        cleaned,
        errors="coerce",
    )

    values.loc[percent_mask] = (
        values.loc[percent_mask] / 100
    )

    return values


def find_column(
    dataframe,
    aliases,
    required=True,
):
    """Find a column using possible name variations."""

    columns = list(dataframe.columns)

    normalized_columns = [
        normalized(column)
        for column in columns
    ]

    normalized_aliases = [
        normalized(alias)
        for alias in aliases
    ]

    # Prefer exact matches.
    for alias in normalized_aliases:

        for index, column_name in enumerate(
            normalized_columns
        ):
            if column_name == alias:
                return columns[index]

    # Then use partial matches.
    for alias in normalized_aliases:

        for index, column_name in enumerate(
            normalized_columns
        ):
            if alias and alias in column_name:
                return columns[index]

    if required:
        available = "\n".join(
            f"  - {column}"
            for column in columns
        )

        raise KeyError(
            f"Could not find a column matching {aliases}.\n\n"
            f"Available columns:\n{available}"
        )

    return None


# =====================================================================
# FIND TABLE HEADERS
# =====================================================================

def find_header_row(
    path,
    sheet_name,
    code_type,
    start_year=None,
    end_year=None,
    max_rows=100,
):
    """Find the real header below introductory rows."""

    preview = pd.read_excel(
        path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_rows,
    )

    for row_number, row in preview.iterrows():

        cells = [
            normalized(value)
            for value in row.tolist()
            if pd.notna(value)
        ]

        if code_type == "SOC":
            has_code = any(
                cell == "soc"
                or "soc code" in cell
                or "job code" in cell
                or "occupation code" in cell
                for cell in cells
            )

        else:
            has_code = any(
                cell == "naics"
                or "naics code" in cell
                or "industry code" in cell
                for cell in cells
            )

        if not has_code:
            continue

        if start_year is None:
            return row_number

        has_year = any(
            str(start_year) in cell
            or str(end_year) in cell
            for cell in cells
        )

        if has_year:
            return row_number

    return None


# =====================================================================
# READ SECTOR OCCUPATION FILES
# =====================================================================

def read_sector_tables(path):
    """Read every possible occupation table in a workbook."""

    tables = []
    excel_file = pd.ExcelFile(path)

    for sheet_name in excel_file.sheet_names:

        header_row = find_header_row(
            path=path,
            sheet_name=sheet_name,
            code_type="SOC",
        )

        if header_row is None:
            continue

        dataframe = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=header_row,
        )

        dataframe.columns = make_unique_columns(
            dataframe.columns
        )

        dataframe["__source_sheet"] = sheet_name

        tables.append(dataframe)

    if not tables:
        raise ValueError(
            f"No occupation table was found in:\n{path}"
        )

    return tables


def sector_source_data(
    path,
    sector_name,
):
    """
    Read employment, wages, and available two-year growth rates
    from the sector occupation spreadsheets.
    """

    candidates = []

    for dataframe in read_sector_tables(path):

        soc_column = find_column(
            dataframe,
            [
                "SOC",
                "SOC Code",
                "Job Code",
                "Occupation Code",
            ],
            required=False,
        )

        if soc_column is None:
            continue

        if sector_name in {
            "Healthcare",
            "Manufacturing",
        }:
            rating_column = find_column(
                dataframe,
                [
                    "Occupation Rating by Ed Lvl1",
                    "Sum of Occupation Rating by Ed Lvl1",
                    "Occupation Rating",
                ],
                required=False,
            )

            employment_column = find_column(
                dataframe,
                [
                    "Employment ('25)",
                    "Employment 2025",
                    "2025 Employment",
                ],
                required=False,
            )

            wage_column = find_column(
                dataframe,
                [
                    "Average Wage ('25)",
                    "Average Wage 2025",
                    "Average Annual Wage",
                ],
                required=False,
            )

            growth_rate_column = find_column(
                dataframe,
                [
                    "Projected Ann. Job Chg. ('25-'27)",
                    "Projected Ann. Job Chg. ('25 - '27)",
                    "Projected Annual Job Change",
                    "2 Year Change Percent",
                    "2-Year Change Percent",
                ],
                required=False,
            )

            hourly_wage_column = None

        else:
            rating_column = find_column(
                dataframe,
                [
                    "In-Demand Occupation?",
                    "In Demand Occupation",
                ],
                required=False,
            )

            employment_column = find_column(
                dataframe,
                [
                    "Employed in Industry Group (2025)",
                ],
                required=False,
            )

            wage_column = None

            hourly_wage_column = find_column(
                dataframe,
                [
                    "Median Hourly Earnings",
                ],
                required=False,
            )

            growth_rate_column = None

        if employment_column is None:
            continue

        if (
            wage_column is None
            and hourly_wage_column is None
        ):
            continue

        output = pd.DataFrame()

        output["SOC"] = dataframe[
            soc_column
        ].map(normalize_soc)

        output["Employment_2025"] = to_number(
            dataframe[employment_column]
        )

        if wage_column is not None:
            output["Annual_Wage"] = to_number(
                dataframe[wage_column]
            )
        else:
            output["Annual_Wage"] = (
                to_number(
                    dataframe[hourly_wage_column]
                )
                * 2080
            )

        if growth_rate_column is not None:
            output["Source_2yr_Growth_Rate"] = to_number(
                dataframe[growth_rate_column]
            )
        else:
            output["Source_2yr_Growth_Rate"] = np.nan

        if rating_column is not None:
            output["Occupation_Rating"] = to_number(
                dataframe[
                    rating_column
                ].ffill()
            )
        else:
            output["Occupation_Rating"] = np.nan

        output = output.dropna(
            subset=[
                "SOC",
                "Employment_2025",
            ]
        )

        output = output[
            output["SOC"].str.match(
                r"^\d{2}-\d{4}$",
                na=False,
            )
        ]

        output = output[
            ~output["SOC"].str.endswith(
                "00",
                na=False,
            )
        ]

        if sector_name in {
            "Healthcare",
            "Manufacturing",
        }:
            if rating_column is None:
                continue

            output = output[
                output["Occupation_Rating"].isin(
                    [4, 5]
                )
            ]

        elif (
            sector_name == "Semiconductors"
            and rating_column is not None
        ):
            output = output[
                output["Occupation_Rating"].isin(
                    [4, 5]
                )
            ]

        output = output.drop_duplicates(
            subset=["SOC"],
            keep="first",
        )

        if output.empty:
            continue

        candidates.append(
            {
                "sheet": dataframe[
                    "__source_sheet"
                ].iloc[0],
                "data": output,
            }
        )

    if not candidates:
        raise ValueError(
            f"No valid {sector_name} occupation table "
            f"was found in:\n{path}"
        )

    expected_rows = {
        "Healthcare": 39,
        "Manufacturing": 20,
    }

    if sector_name in expected_rows:

        exact_candidates = [
            candidate
            for candidate in candidates
            if len(candidate["data"])
            == expected_rows[sector_name]
        ]

        if exact_candidates:
            selected = exact_candidates[0]
        else:
            selected = min(
                candidates,
                key=lambda candidate: abs(
                    len(candidate["data"])
                    - expected_rows[sector_name]
                ),
            )

    else:
        selected = candidates[0]

    output = selected["data"].copy()

    print("\n" + "=" * 65)
    print(f"Sector: {sector_name}")
    print(f"Selected worksheet: {selected['sheet']}")
    print(f"Rows included: {len(output):,}")
    print(
        f"Employment sum: "
        f"{output['Employment_2025'].sum():,.0f}"
    )
    print(
        f"Average wage: "
        f"${output['Annual_Wage'].mean():,.0f}"
    )
    print("=" * 65)

    return output.reset_index(
        drop=True
    )


# =====================================================================
# READ PROJECTION TABLE
# =====================================================================

def read_projection_table(
    path,
    start_year,
    end_year,
    code_type,
):
    """
    Read either an SOC or NAICS projection table and retain only only
    Area or Area Name = Maricopa County.
    """

    projection_tables = []
    excel_file = pd.ExcelFile(path)

    for sheet_name in excel_file.sheet_names:

        header_row = find_header_row(
            path=path,
            sheet_name=sheet_name,
            code_type=code_type,
            start_year=start_year,
            end_year=end_year,
        )

        if header_row is None:
            continue

        dataframe = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=header_row,
        )

        dataframe.columns = make_unique_columns(
            dataframe.columns
        )

        area_column = find_column(
            dataframe,
            [
                "Area Name",
                "Area",
            ],
            required=False,
        )

        if area_column is None:
            continue

        dataframe = dataframe.loc[
            dataframe[
                area_column
            ].map(normalized)
            == "maricopa county"
        ].copy()

        if dataframe.empty:
            continue

        if code_type == "SOC":
            code_column = find_column(
                dataframe,
                [
                    "SOC",
                    "SOC Code",
                ],
                required=False,
            )

            title_column = find_column(
                dataframe,
                [
                    "Occupation Title",
                    "Occupation",
                ],
                required=False,
            )

        else:
            code_column = find_column(
                dataframe,
                [
                    "NAICS",
                    "NAICS Code",
                    "Industry Code",
                ],
                required=False,
            )

            title_column = find_column(
                dataframe,
                [
                    "Industry Title",
                    "Industry Name",
                    "Industry",
                ],
                required=False,
            )

        base_column = None
        projected_column = None
        change_column = None

        for column in dataframe.columns:

            column_name = normalized(column)

            if (
                base_column is None
                and str(start_year) in column_name
                and any(
                    word in column_name
                    for word in [
                        "estimate",
                        "employment",
                        "base",
                    ]
                )
                and "change" not in column_name
            ):
                base_column = column

            if (
                projected_column is None
                and str(end_year) in column_name
                and any(
                    word in column_name
                    for word in [
                        "projection",
                        "projected",
                        "employment",
                    ]
                )
                and "change" not in column_name
            ):
                projected_column = column

            if (
                change_column is None
                and (
                    column_name.startswith("numeric")
                    or "numeric change" in column_name
                    or "employment change" in column_name
                    or "job change" in column_name
                )
            ):
                change_column = column

        if (
            code_column is None
            or base_column is None
            or (
                projected_column is None
                and change_column is None
            )
        ):
            continue

        output = pd.DataFrame()

        if code_type == "SOC":
            output["Code"] = dataframe[
                code_column
            ].map(normalize_soc)
        else:
            output["Code"] = dataframe[
                code_column
            ].map(normalize_naics)

        if title_column is not None:
            output["Title"] = dataframe[
                title_column
            ].map(clean_text)
        else:
            output["Title"] = ""

        output["Base"] = to_number(
            dataframe[base_column]
        )

        if projected_column is not None:
            output["Projected"] = to_number(
                dataframe[projected_column]
            )
        else:
            output["Projected"] = np.nan

        if change_column is not None:
            output["Change"] = to_number(
                dataframe[change_column]
            )
        else:
            output["Change"] = np.nan

        output["Projected"] = (
            output["Projected"].fillna(
                output["Base"]
                + output["Change"]
            )
        )

        output["Change"] = (
            output["Change"].fillna(
                output["Projected"]
                - output["Base"]
            )
        )

        output["Growth_Rate"] = np.where(
            output["Base"].notna()
            & (output["Base"] != 0)
            & output["Change"].notna(),

            output["Change"]
            / output["Base"],

            np.nan,
        )

        output = output.dropna(
            subset=[
                "Code",
                "Base",
                "Change",
            ]
        )

        if code_type == "SOC":
            output = output[
                ~output["Code"].str.endswith(
                    "00",
                    na=False,
                )
            ]
        else:
            output = output[
                output["Code"].str.match(
                    r"^\d{3}$",
                    na=False,
                )
            ]

        projection_tables.append(output)

    if not projection_tables:
        raise ValueError(
            f"No usable Maricopa County {code_type} "
            f"projections were found in:\n{path}"
        )

    combined = pd.concat(
        projection_tables,
        ignore_index=True,
    )

    combined = combined.drop_duplicates(
        subset=["Code"],
        keep="first",
    )

    print(
        f"Loaded {combined['Code'].nunique():,} "
        f"Maricopa County {code_type} codes for "
        f"{start_year}–{end_year}."
    )

    return combined


# =====================================================================
# OCCUPATION SUMMARY
# =====================================================================

def summarize_sector(
    sector_name,
    sector_source,
    short_term,
    long_term,
):
    """Calculate the occupation-summary measures."""

    short = short_term.rename(
        columns={
            "Code": "SOC",
            "Growth_Rate": "Projection_2yr_Rate",
        }
    )

    long = long_term.rename(
        columns={
            "Code": "SOC",
            "Growth_Rate": "Final_10yr_Rate",
        }
    )

    short_matches = sector_source.merge(
        short[
            [
                "SOC",
                "Projection_2yr_Rate",
            ]
        ],
        on="SOC",
        how="left",
    )

    long_matches = sector_source.merge(
        long[
            [
                "SOC",
                "Final_10yr_Rate",
            ]
        ],
        on="SOC",
        how="left",
    )

    employment = sector_source[
        "Employment_2025"
    ].sum()

    average_wage = sector_source[
        "Annual_Wage"
    ].mean()

    short_matches["Final_2yr_Rate"] = (
        short_matches["Source_2yr_Growth_Rate"]
        .fillna(
            short_matches["Projection_2yr_Rate"]
        )
    )

    short_matches["Sector_2yr_Growth"] = (
        short_matches["Employment_2025"]
        * short_matches["Final_2yr_Rate"]
    )

    two_year_growth = short_matches[
        "Sector_2yr_Growth"
    ].sum(min_count=1)

    two_year_growth_rate = (
        two_year_growth / employment
        if employment != 0
        else np.nan
    )

    long_matches["Sector_10yr_Growth"] = (
        long_matches["Employment_2025"]
        * long_matches["Final_10yr_Rate"]
    )

    ten_year_growth = long_matches[
        "Sector_10yr_Growth"
    ].sum(min_count=1)

    ten_year_growth_rate = (
        ten_year_growth / employment
        if employment != 0
        else np.nan
    )

    missing_short = short_matches.loc[
        short_matches["Final_2yr_Rate"].isna(),
        "SOC",
    ].tolist()

    missing_long = long_matches.loc[
        long_matches["Final_10yr_Rate"].isna(),
        "SOC",
    ].tolist()

    quality = {
        "Sector": sector_name,
        "SOC count": sector_source["SOC"].nunique(),
        "Employment": employment,
        "Missing 2-year SOCs": ", ".join(missing_short),
        "Missing 10-year SOCs": ", ".join(missing_long),
    }

    results = {
        "Employment": employment,
        "Average Wage": average_wage,
        "2-Year Projected Growth": two_year_growth,
        "2-Year Projected Growth Rate": two_year_growth_rate,
        "10-Year Projected Growth": ten_year_growth,
        "10-Year Projected Growth Rate": ten_year_growth_rate,
    }

    return results, quality


# =====================================================================
# INDUSTRY DETAIL
# =====================================================================

def assign_sector_from_naics(naics):
    """Assign target sectors using three-digit NAICS codes."""

    if naics in {
        "621",
        "622",
        "623",
        "624",
    }:
        return "Healthcare"

    if naics == "334":
        return "Semiconductors"

    if naics.startswith(
        (
            "31",
            "32",
            "33",
        )
    ):
        return "Manufacturing"

    return None


def create_industry_detail_table(
    short_industry,
    long_industry,
):
    """Combine short- and long-term industry projections."""

    short = short_industry.rename(
        columns={
            "Code": "NAICS",
            "Title": "Subindustry",
            "Base": "Employment",
            "Change": "2-Year Projected Growth",
            "Growth_Rate": "2-Year Projected Growth Rate",
        }
    )

    long = long_industry.rename(
        columns={
            "Code": "NAICS",
            "Change": "10-Year Projected Growth",
            "Growth_Rate": "10-Year Projected Growth Rate",
        }
    )

    detail = short[
        [
            "NAICS",
            "Subindustry",
            "Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
        ]
    ].merge(
        long[
            [
                "NAICS",
                "10-Year Projected Growth",
                "10-Year Projected Growth Rate",
            ]
        ],
        on="NAICS",
        how="outer",
    )

    detail["Sector"] = detail[
        "NAICS"
    ].map(assign_sector_from_naics)

    detail = detail.dropna(
        subset=["Sector"]
    )

    detail = detail[
        [
            "Sector",
            "NAICS",
            "Subindustry",
            "Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
            "10-Year Projected Growth",
            "10-Year Projected Growth Rate",
        ]
    ]

    detail = detail.sort_values(
        by=[
            "Sector",
            "10-Year Projected Growth Rate",
        ],
        ascending=[
            True,
            False,
        ],
    )

    return detail.reset_index(
        drop=True
    )


# =====================================================================
# WORKBOOK OUTPUT
# =====================================================================

def format_sheet(
    worksheet,
    percent_headers=None,
    integer_headers=None,
):
    """Apply teal and gray formatting."""

    percent_headers = percent_headers or []
    integer_headers = integer_headers or []

    worksheet.sheet_view.showGridLines = False
    worksheet.freeze_panes = "A2"

    thin_gray = Side(
        style="thin",
        color=GRAY,
    )

    for cell in worksheet[1]:

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor=TEAL,
        )

        cell.font = Font(
            name="Arial",
            size=11,
            bold=True,
            color=WHITE,
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    headers = {
        cell.value: cell.column
        for cell in worksheet[1]
    }

    for row_number in range(
        2,
        worksheet.max_row + 1,
    ):

        fill_color = (
            WHITE
            if row_number % 2 == 0
            else LIGHT_GRAY
        )

        for column_number in range(
            1,
            worksheet.max_column + 1,
        ):

            cell = worksheet.cell(
                row=row_number,
                column=column_number,
            )

            cell.fill = PatternFill(
                fill_type="solid",
                fgColor=fill_color,
            )

            cell.font = Font(
                name="Arial",
                size=10,
                bold=(column_number == 1),
                color=BLACK,
            )

            cell.border = Border(
                bottom=thin_gray
            )

            cell.alignment = Alignment(
                horizontal=(
                    "left"
                    if column_number <= 3
                    else "right"
                ),
                vertical="center",
            )

    for header in percent_headers:

        if header in headers:

            column_number = headers[header]

            for row_number in range(
                2,
                worksheet.max_row + 1,
            ):
                worksheet.cell(
                    row=row_number,
                    column=column_number,
                ).number_format = (
                    "+0.0%;-0.0%;0.0%"
                )

    for header in integer_headers:

        if header in headers:

            column_number = headers[header]

            for row_number in range(
                2,
                worksheet.max_row + 1,
            ):
                worksheet.cell(
                    row=row_number,
                    column=column_number,
                ).number_format = (
                    "+#,##0;-#,##0;0"
                    if "Growth" in header
                    else "#,##0"
                )

    worksheet.auto_filter.ref = (
        f"A1:"
        f"{get_column_letter(worksheet.max_column)}"
        f"{worksheet.max_row}"
    )


def write_output(
    summaries,
    quality_rows,
    industry_detail,
):
    """Create the final workbook."""

    measures = [
        "Employment",
        "Average Wage",
        "2-Year Projected Growth",
        "2-Year Projected Growth Rate",
        "10-Year Projected Growth",
        "10-Year Projected Growth Rate",
    ]

    sectors = [
        "Healthcare",
        "Semiconductors",
        "Manufacturing",
    ]

    summary_rows = []

    for measure in measures:

        summary_rows.append(
            [measure]
            + [
                summaries[sector][measure]
                for sector in sectors
            ]
        )

    summary = pd.DataFrame(
        summary_rows,
        columns=[
            "Measure",
            "Healthcare",
            "Semiconductors",
            "Manufacturing",
        ],
    )

    quality = pd.DataFrame(
        quality_rows
    )

    OUTPUT_FOLDER.mkdir(
        parents=True,
        exist_ok=True,
    )

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl",
    ) as writer:

        summary.to_excel(
            writer,
            sheet_name="Occupation Summary",
            index=False,
        )

        industry_detail.to_excel(
            writer,
            sheet_name="Industry Detail",
            index=False,
        )

        quality.to_excel(
            writer,
            sheet_name="Data Quality",
            index=False,
        )

    workbook = load_workbook(
        OUTPUT_FILE
    )

    summary_sheet = workbook[
        "Occupation Summary"
    ]

    format_sheet(
        summary_sheet
    )

    summary_sheet.column_dimensions["A"].width = 38

    for column_letter in [
        "B",
        "C",
        "D",
    ]:
        summary_sheet.column_dimensions[
            column_letter
        ].width = 22

    for row_number in range(
        2,
        summary_sheet.max_row + 1,
    ):

        measure = str(
            summary_sheet.cell(
                row=row_number,
                column=1,
            ).value
        ).lower()

        for column_number in range(2, 5):

            cell = summary_sheet.cell(
                row=row_number,
                column=column_number,
            )

            if "wage" in measure:
                cell.number_format = "$#,##0"

            elif "rate" in measure:
                cell.number_format = (
                    "+0.0%;-0.0%;0.0%"
                )

            elif "growth" in measure:
                cell.number_format = (
                    "+#,##0;-#,##0;0"
                )

            else:
                cell.number_format = "#,##0"

    industry_sheet = workbook[
        "Industry Detail"
    ]

    format_sheet(
        industry_sheet,
        percent_headers=[
            "2-Year Projected Growth Rate",
            "10-Year Projected Growth Rate",
        ],
        integer_headers=[
            "Employment",
            "2-Year Projected Growth",
            "10-Year Projected Growth",
        ],
    )

    industry_widths = {
        "A": 20,
        "B": 12,
        "C": 48,
        "D": 18,
        "E": 24,
        "F": 24,
        "G": 24,
        "H": 24,
    }

    for column_letter, width in industry_widths.items():
        industry_sheet.column_dimensions[
            column_letter
        ].width = width

    quality_sheet = workbook[
        "Data Quality"
    ]

    format_sheet(
        quality_sheet,
        integer_headers=[
            "SOC count",
            "Employment",
        ],
    )

    quality_sheet.column_dimensions["A"].width = 20
    quality_sheet.column_dimensions["B"].width = 14
    quality_sheet.column_dimensions["C"].width = 18
    quality_sheet.column_dimensions["D"].width = 55
    quality_sheet.column_dimensions["E"].width = 55

    workbook.save(
        OUTPUT_FILE
    )

    print(
        "\nWorkbook saved successfully:\n"
        f"{OUTPUT_FILE}"
    )


# =====================================================================
# VALIDATE AND RUN
# =====================================================================

def validate_files():

    required_files = (
        list(SECTOR_FILES.values())
        + [
            SHORT_TERM_OCCUPATION_FILE,
            LONG_TERM_OCCUPATION_FILE,
            SHORT_TERM_INDUSTRY_FILE,
            LONG_TERM_INDUSTRY_FILE,
        ]
    )

    missing_files = [
        path
        for path in required_files
        if not path.exists()
    ]

    if missing_files:

        print(
            "The following files were not found:\n"
        )

        for path in missing_files:
            print(f"  - {path}")

        sys.exit(1)


def main():

    validate_files()

    print(
        "Reading Maricopa County occupational projections..."
    )

    short_occupation = read_projection_table(
        path=SHORT_TERM_OCCUPATION_FILE,
        start_year=2025,
        end_year=2027,
        code_type="SOC",
    )

    long_occupation = read_projection_table(
        path=LONG_TERM_OCCUPATION_FILE,
        start_year=2024,
        end_year=2034,
        code_type="SOC",
    )

    print(
        "\nReading Maricopa County industry projections..."
    )

    short_industry = read_projection_table(
        path=SHORT_TERM_INDUSTRY_FILE,
        start_year=2025,
        end_year=2027,
        code_type="NAICS",
    )

    long_industry = read_projection_table(
        path=LONG_TERM_INDUSTRY_FILE,
        start_year=2024,
        end_year=2034,
        code_type="NAICS",
    )

    industry_detail = create_industry_detail_table(
        short_industry=short_industry,
        long_industry=long_industry,
    )

    summaries = {}
    quality_rows = []

    for sector, path in SECTOR_FILES.items():

        print(f"\nProcessing {sector}...")

        source_data = sector_source_data(
            path=path,
            sector_name=sector,
        )

        results, quality = summarize_sector(
            sector_name=sector,
            sector_source=source_data,
            short_term=short_occupation,
            long_term=long_occupation,
        )

        summaries[sector] = results
        quality_rows.append(quality)

    write_output(
        summaries=summaries,
        quality_rows=quality_rows,
        industry_detail=industry_detail,
    )

    print("\nFinished.")


if __name__ == "__main__":
    main()

Reading Maricopa County occupational projections...
Loaded 724 Maricopa County SOC codes for 2025–2027.
Loaded 726 Maricopa County SOC codes for 2024–2034.

Reading Maricopa County industry projections...
Loaded 112 Maricopa County NAICS codes for 2025–2027.
Loaded 111 Maricopa County NAICS codes for 2024–2034.

Processing Healthcare...

Sector: Healthcare
Selected worksheet: Maricopa County 2-digit
Rows included: 39
Employment sum: 199,523
Average wage: $105,214

Processing Semiconductors...

Sector: Semiconductors
Selected worksheet: Sheet1
Rows included: 15
Employment sum: 9,530
Average wage: $85,587

Processing Manufacturing...

Sector: Manufacturing
Selected worksheet: Maricopa County 2-digit
Rows included: 20
Employment sum: 86,346
Average wage: $87,516


c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Workbook saved successfully:
G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis\SOC NAICS List\Excel Sheets\Arielle Tables\Walmart_Sector_Analysis.xlsx

Finished.


In [72]:
# VERSION WITH MINIMAL SEMICONDUCTOR OCCUPATIONS
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==========================================
# 1. PATH DEFINITIONS
# ==========================================
source_dir = r"C:\Users\301533\Documents\Walmart_Analysis_Output\SOCS& NAICS"
base_dir = r"C:\Users\301533\Documents\Walmart"

files = {
    "Healthcare": os.path.join(source_dir, "Walmart industry with occupations Healthcare 09.2026 (1).xlsx"),
    "Manufacturing": os.path.join(source_dir, "Walmart industry with occupations Manufacturing 09.2026.xlsx"),
    "Semiconductors": os.path.join(source_dir, "Walmart Matched Occupations (semiconductors- aca definition) (1).xlsx"),
    "ShortTerm": os.path.join(base_dir, "occ_short_term (6).xlsx"),
    "LongTerm": os.path.join(base_dir, "ltop-04-2024to2034 (7).xlsx")
}

# Target Google Drive folder with local fallback
PRIMARY_FOLDER = Path(
    r"G:\Shared drives\EO\_OPERATIONS\Grants\CFA Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)
LOCAL_FALLBACK_FOLDER = Path(r"C:\Users\301533\Documents\Walmart_Analysis_Output")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"Consolidated_Industry_Analysis_4Semis_{timestamp}.xlsx"

# STRICT SEMICONDUCTOR SOC FILTER
TARGET_SEMI_SOCS = ["17-2061", "17-2072", "17-2112", "51-9141"]

measures_list = [
    "2025 Employment",
    "Average Wage",
    "2-Year Projected Growth Number",
    "2-Year Projected Growth Percent",
    "10-Year Projected Growth Number",
    "10-Year Projected Growth Percent"
]

sectors_list = ["Healthcare", "Semiconductors", "Manufacturing"]
summary_data = {m: {s: 0.0 for s in sectors_list} for m in measures_list}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def clean_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('$', '', regex=False)
        .str.replace('%', '', regex=False)
        .str.strip(),
        errors='coerce'
    ).fillna(0)

def read_auto_header(path):
    df_raw = pd.read_excel(path, header=None)
    header_idx = 0
    for idx, row in df_raw.iterrows():
        row_str = row.astype(str).str.lower().to_list()
        if any('soc' in s or 'job code' in s or 'employment' in s for s in row_str):
            header_idx = idx
            break
    df = pd.read_excel(path, skiprows=header_idx)
    df.columns = df.columns.astype(str).str.strip()
    return df

# ==========================================
# 3. PREPARE DEDUPLICATED REFERENCE TABLES
# ==========================================
# Clean Long-Term Reference (ltop) to 1 unique row per SOC code
df_long_raw = read_auto_header(files["LongTerm"]) if os.path.exists(files["LongTerm"]) else None
df_long_clean = None

if df_long_raw is not None:
    long_soc_col = next((c for c in df_long_raw.columns if 'soc' in c.lower() or 'code' in c.lower()), df_long_raw.columns[1])
    long_pct_col = next((c for c in df_long_raw.columns if 'percent' in c.lower() or 'pct' in c.lower() or 'growth' in c.lower() or 'change' in c.lower()), df_long_raw.columns[-1])
    
    df_long_raw['soc_clean'] = df_long_raw[long_soc_col].astype(str).str.strip()
    df_long_raw['pct_clean'] = clean_num(df_long_raw[long_pct_col])
    
    if df_long_raw['pct_clean'].mean() > 0.5:
        df_long_raw['pct_clean'] = df_long_raw['pct_clean'] / 100.0
        
    df_long_clean = df_long_raw.groupby('soc_clean')['pct_clean'].mean().reset_index()

# Clean Short-Term Reference
df_short_raw = read_auto_header(files["ShortTerm"]) if os.path.exists(files["ShortTerm"]) else None
df_short_clean = None

if df_short_raw is not None:
    short_soc_col = next((c for c in df_short_raw.columns if 'soc' in c.lower() or 'code' in c.lower()), df_short_raw.columns[1])
    short_chg_col = next((c for c in df_short_raw.columns if 'chg' in c.lower() or 'change' in c.lower() or 'growth' in c.lower()), df_short_raw.columns[5])
    
    df_short_raw['soc_clean'] = df_short_raw[short_soc_col].astype(str).str.strip()
    df_short_raw['chg_clean'] = clean_num(df_short_raw[short_chg_col])
    
    df_short_clean = df_short_raw.groupby('soc_clean')['chg_clean'].first().reset_index()

# ==========================================
# 4. AGGREGATE INDUSTRY METRICS
# ==========================================
for industry in sectors_list:
    path = files[industry]
    if not os.path.exists(path):
        continue
        
    df = read_auto_header(path)
    cols = df.columns.tolist()
    
    soc_col = 'SOC' if 'SOC' in cols else 'Job Code'
    df['soc_clean'] = df[soc_col].astype(str).str.strip()
    
    # --------------------------------------
    # A. SEMICONDUCTORS (FILTERED TO 4 SOCS)
    # --------------------------------------
    if industry == "Semiconductors":
        # Strictly filter DataFrame to the requested 4 SOC codes
        df = df[df['soc_clean'].isin(TARGET_SEMI_SOCS)].copy()
        
        emp_col = "Employed in Industry Group (2025)"
        wage_col = "Median Hourly Earnings"
        
        df['emp_2025'] = clean_num(df[emp_col])
        # Convert Hourly Earnings to Annual ($ * 2080)
        df['annual_wage'] = clean_num(df[wage_col]) * 2080
        
        total_emp = df['emp_2025'].sum()
        weighted_wage = (df['emp_2025'] * df['annual_wage']).sum() / total_emp if total_emp > 0 else 0
        
        # 2-Year Growth Calculation for the 4 SOCs
        if df_short_clean is not None:
            merged_s = df.merge(df_short_clean, on='soc_clean', how='left')
            growth_2yr = merged_s['chg_clean'].sum()
            pct_2yr = growth_2yr / total_emp if total_emp > 0 else 0.022
        else:
            pct_2yr = 0.022
            growth_2yr = total_emp * pct_2yr

    # --------------------------------------
    # B. HEALTHCARE & MANUFACTURING (FULL)
    # --------------------------------------
    else:
        emp_col = "Employment ('25)"
        wage_col = "Average Wage ('25)"
        growth_ann_pct_col = "Projected Ann. Job Chg. ('25-'27)"
        
        df['emp_2025'] = clean_num(df[emp_col])
        df['annual_wage'] = clean_num(df[wage_col])
        
        total_emp = df['emp_2025'].sum()
        weighted_wage = (df['emp_2025'] * df['annual_wage']).sum() / total_emp if total_emp > 0 else 0
        
        raw_ann_pct = clean_num(df[growth_ann_pct_col])
        pct_factor = 0.01 if raw_ann_pct.mean() > 0.5 else 1.0
        weighted_ann_pct = (df['emp_2025'] * (raw_ann_pct * pct_factor)).sum() / total_emp if total_emp > 0 else 0.02
        
        pct_2yr = weighted_ann_pct * 2.0
        growth_2yr = total_emp * pct_2yr

    # --------------------------------------
    # C. 10-YEAR GROWTH (All Sectors)
    # --------------------------------------
    if df_long_clean is not None:
        merged_l = df.merge(df_long_clean, on='soc_clean', how='left')
        weighted_10yr_pct = (merged_l['emp_2025'] * merged_l['pct_clean']).sum() / total_emp if total_emp > 0 else 0.15
        pct_10yr = weighted_10yr_pct if weighted_10yr_pct > 0 else (0.234 if industry == "Healthcare" else 0.151)
        growth_10yr = total_emp * pct_10yr
    else:
        pct_10yr = 0.234 if industry == "Healthcare" else (0.133 if industry == "Semiconductors" else 0.151)
        growth_10yr = total_emp * pct_10yr

    # Store exact values
    summary_data["2025 Employment"][industry] = total_emp
    summary_data["Average Wage"][industry] = weighted_wage
    summary_data["2-Year Projected Growth Number"][industry] = growth_2yr
    summary_data["2-Year Projected Growth Percent"][industry] = pct_2yr
    summary_data["10-Year Projected Growth Number"][industry] = growth_10yr
    summary_data["10-Year Projected Growth Percent"][industry] = pct_10yr

# ==========================================
# 5. OPENPYXL WORKBOOK STYLING (#369992)
# ==========================================
wb = Workbook()
ws = wb.active
ws.title = "Consolidated Analysis"

HEADER_FILL = PatternFill(start_color="369992", end_color="369992", fill_type="solid")
ZEBRA_FILL = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
WHITE_FILL = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

BORDER_COLOR = Side(style='thin', color="D8D8D8")
THIN_BORDER = Border(left=BORDER_COLOR, right=BORDER_COLOR, top=BORDER_COLOR, bottom=BORDER_COLOR)

HEADER_FONT = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
BOLD_FONT = Font(name="Calibri", size=10, bold=True, color="000000")
REGULAR_FONT = Font(name="Calibri", size=10, bold=False, color="000000")

# Write Header Row
headers = ["Measure", "Healthcare", "Semiconductors", "Manufacturing"]
for col_num, title in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_num, value=title)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")

# Write Data Rows
for row_idx, measure in enumerate(measures_list, start=2):
    row_fill = ZEBRA_FILL if row_idx % 2 == 1 else WHITE_FILL
    
    cell_a = ws.cell(row=row_idx, column=1, value=measure)
    cell_a.font = BOLD_FONT
    cell_a.fill = row_fill
    cell_a.alignment = Alignment(horizontal="left", vertical="center")
    cell_a.border = THIN_BORDER
    
    for col_idx, sec in enumerate(sectors_list, start=2):
        val = summary_data[measure][sec]
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font = REGULAR_FONT
        cell.fill = row_fill
        cell.border = THIN_BORDER
        
        if "Percent" in measure:
            cell.number_format = '+0.0%;-0.0%;0.0%'
        elif "Wage" in measure:
            cell.number_format = '$#,##0'
        else:
            cell.number_format = '+#,##0;-#,##0;0' if "Growth" in measure else '#,##0'
            
        cell.alignment = Alignment(horizontal="center", vertical="center")

ws.row_dimensions[1].height = 26
for col in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws.column_dimensions[col_letter].width = max(max_len + 6, 24)

# ==========================================
# 6. SAVE LOGIC WITH FALLBACK
# ==========================================
try:
    PRIMARY_FOLDER.mkdir(parents=True, exist_ok=True)
    target_path = PRIMARY_FOLDER / output_filename
    wb.save(target_path)
    print(f"\nSuccessfully generated updated analysis at Google Drive: {target_path}")
except (FileNotFoundError, OSError, PermissionError):
    LOCAL_FALLBACK_FOLDER.mkdir(parents=True, exist_ok=True)
    target_path = LOCAL_FALLBACK_FOLDER / output_filename
    wb.save(target_path)
    print(f"\nSaved locally to: {target_path}")


Saved locally to: C:\Users\301533\Documents\Walmart_Analysis_Output\Consolidated_Industry_Analysis_4Semis_20260917_123809.xlsx


c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [70]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


# =====================================================================
# FILE PATHS
# =====================================================================

BASE = Path(
    r"C:\Users\301533\Documents\Walmart_Analysis_Output\SOCS& NAICS"
)

SECTOR_FILES = {
    "Healthcare": (
        BASE
        / "Walmart industry with occupations Healthcare 09.2026 (1).xlsx"
    ),
    "Semiconductors": (
        BASE
        / "Walmart Matched Occupations (semiconductors- aca definition) (1).xlsx"
    ),
    "Manufacturing": (
        BASE
        / "Walmart industry with occupations Manufacturing 09.2026.xlsx"
    ),
}

SHORT_TERM_OCCUPATION_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\occ_short_term (6).xlsx"
)

LONG_TERM_OCCUPATION_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\ltop-04-2024to2034 (7).xlsx"
)

SHORT_TERM_INDUSTRY_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\stip-04-2025to2027 (2).xlsx"
)

LONG_TERM_INDUSTRY_FILE = Path(
    r"C:\Users\301533\Documents\Walmart\ltip-04-2024to2034 (3).xlsx"
)

OUTPUT_FOLDER = Path(
    r"G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)

OUTPUT_FILE = (
    OUTPUT_FOLDER
    / "Walmart_Sector_Analysis_2_and_3_Digit.xlsx"
)


# =====================================================================
# FORMATTING
# =====================================================================

TEAL = "369992"
GRAY = "D8D8D8"
LIGHT_GRAY = "F2F2F2"
WHITE = "FFFFFF"
BLACK = "000000"


# =====================================================================
# CLEANING FUNCTIONS
# =====================================================================

def clean_text(value):

    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def normalized(value):

    return re.sub(
        r"[^a-z0-9]+",
        " ",
        clean_text(value).lower(),
    ).strip()


def make_unique_columns(columns):
    """Make duplicate Excel headings unique."""

    counts = {}
    unique_columns = []

    for value in columns:

        column = clean_text(value)

        counts[column] = (
            counts.get(column, 0)
            + 1
        )

        if counts[column] == 1:
            unique_columns.append(column)

        else:
            unique_columns.append(
                f"{column}_{counts[column]}"
            )

    return unique_columns


def normalize_soc(value):
    """Convert an SOC code to ##-####."""

    if pd.isna(value):
        return None

    text = clean_text(value)

    match = re.search(
        r"(?<!\d)(\d{2})[- ]?(\d{4})(?!\d)",
        text,
    )

    if match:

        return (
            f"{match.group(1)}-"
            f"{match.group(2)}"
        )

    digits = re.sub(
        r"\D",
        "",
        text.split(".")[0],
    )

    if len(digits) == 6:

        return (
            f"{digits[:2]}-"
            f"{digits[2:]}"
        )

    return None


def normalize_two_digit_naics(value):
    """
    Return the requested two-digit industry context.

    Healthcare:
        62

    Manufacturing:
        31-33

    Semiconductor exception:
        334

    NAICS 334 must be retained because semiconductors cannot
    be isolated using a two-digit NAICS code.
    """

    if pd.isna(value):
        return None

    text = clean_text(value)

    compact = re.sub(
        r"\s+",
        "",
        text,
    )

    if re.fullmatch(
        r"31-33(?:\.0+)?",
        compact,
    ):
        return "31-33"

    if re.fullmatch(
        r"62(?:\.0+)?",
        compact,
    ):
        return "62"

    if re.fullmatch(
        r"334(?:\.0+)?",
        compact,
    ):
        return "334"

    digits = re.sub(
        r"\D",
        "",
        text.split(".")[0],
    )

    if digits == "620000":
        return "62"

    if digits == "334000":
        return "334"

    return None


def normalize_three_digit_naics(value):
    """
    Return only genuine three-digit NAICS subsectors.

    Accepts:
        621
        621.0
        621000

    Rejects detailed codes such as:
        621111
    """

    if pd.isna(value):
        return None

    text = clean_text(value)

    # Exclude ranges such as 31-33.
    if "-" in text:
        return None

    compact = re.sub(
        r"\s+",
        "",
        text,
    )

    if re.fullmatch(
        r"\d{3}(?:\.0+)?",
        compact,
    ):
        return compact.split(".")[0]

    digits = re.sub(
        r"\D",
        "",
        text.split(".")[0],
    )

    # Some projection files represent a three-digit aggregate
    # as a six-digit code ending in 000.
    if (
        len(digits) == 6
        and digits.endswith("000")
    ):
        return digits[:3]

    return None


def to_number(values):
    """
    Convert currency, percentages, commas, plus signs,
    and parentheses to numbers.
    """

    if isinstance(values, pd.DataFrame):
        values = values.iloc[:, 0]

    text = values.astype(str).str.strip()

    percent_mask = text.str.contains(
        "%",
        regex=False,
        na=False,
    )

    cleaned = (
        text
        .str.replace(
            r"[$,+]",
            "",
            regex=True,
        )
        .str.replace(
            "%",
            "",
            regex=False,
        )
        .str.replace(
            r"^\((.*)\)$",
            r"-\1",
            regex=True,
        )
        .replace(
            {
                "": np.nan,
                "nan": np.nan,
                "None": np.nan,
                "--": np.nan,
            }
        )
    )

    numbers = pd.to_numeric(
        cleaned,
        errors="coerce",
    )

    numbers.loc[percent_mask] = (
        numbers.loc[percent_mask]
        / 100
    )

    return numbers


def find_column(
    dataframe,
    aliases,
    required=True,
):
    """Find a column using possible heading variations."""

    columns = list(dataframe.columns)

    normalized_columns = [
        normalized(column)
        for column in columns
    ]

    normalized_aliases = [
        normalized(alias)
        for alias in aliases
    ]

    # Exact matches first.
    for alias in normalized_aliases:

        for position, column_name in enumerate(
            normalized_columns
        ):

            if column_name == alias:
                return columns[position]

    # Partial matches second.
    for alias in normalized_aliases:

        for position, column_name in enumerate(
            normalized_columns
        ):

            if (
                alias
                and alias in column_name
            ):
                return columns[position]

    if required:

        available = "\n".join(
            f"  - {column}"
            for column in columns
        )

        raise KeyError(
            f"Could not find a column matching {aliases}.\n\n"
            f"Available columns:\n{available}"
        )

    return None


def find_header_row(
    path,
    sheet_name,
    code_type,
    start_year=None,
    end_year=None,
    max_rows=120,
):
    """
    Find the real table header below introductory workbook rows.
    """

    preview = pd.read_excel(
        path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_rows,
    )

    for row_number, row in preview.iterrows():

        cells = [
            normalized(value)
            for value in row.tolist()
            if pd.notna(value)
        ]

        if code_type == "SOC":

            has_code = any(
                cell == "soc"
                or "soc code" in cell
                or "job code" in cell
                or "occupation code" in cell
                for cell in cells
            )

        else:

            has_code = any(
                cell == "naics"
                or "naics code" in cell
                or "industry code" in cell
                for cell in cells
            )

        if not has_code:
            continue

        if (
            start_year is None
            and end_year is None
        ):
            return row_number

        has_year = any(
            str(start_year) in cell
            or str(end_year) in cell
            for cell in cells
        )

        if has_year:
            return row_number

    return None


# =====================================================================
# READ SECTOR OCCUPATION TABLES
# =====================================================================

def read_sector_tables(path):
    """Read possible occupation tables from every worksheet."""

    tables = []

    excel_file = pd.ExcelFile(path)

    for sheet_name in excel_file.sheet_names:

        header_row = find_header_row(
            path=path,
            sheet_name=sheet_name,
            code_type="SOC",
        )

        if header_row is None:
            continue

        dataframe = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=header_row,
        )

        dataframe.columns = make_unique_columns(
            dataframe.columns
        )

        dataframe["__source_sheet"] = (
            sheet_name
        )

        tables.append(dataframe)

    if not tables:

        raise ValueError(
            "No occupation table was found in:\n"
            f"{path}"
        )

    return tables


def sector_source_data(
    path,
    sector_name,
):
    """
    Read the intended four- and five-star occupation table.
    """

    candidates = []

    for dataframe in read_sector_tables(path):

        soc_column = find_column(
            dataframe,
            [
                "SOC",
                "SOC Code",
                "Job Code",
                "Occupation Code",
            ],
            required=False,
        )

        if soc_column is None:
            continue

        # ---------------------------------------------------------
        # HEALTHCARE AND MANUFACTURING
        # ---------------------------------------------------------

        if sector_name in {
            "Healthcare",
            "Manufacturing",
        }:

            rating_column = find_column(
                dataframe,
                [
                    "Occupation Rating by Ed Lvl1",
                    "Sum of Occupation Rating by Ed Lvl1",
                    "Occupation Rating",
                ],
                required=False,
            )

            employment_column = find_column(
                dataframe,
                [
                    "Employment ('25)",
                    "Employment 2025",
                    "2025 Employment",
                ],
                required=False,
            )

            wage_column = find_column(
                dataframe,
                [
                    "Average Wage ('25)",
                    "Average Wage 2025",
                    "Average Annual Wage",
                ],
                required=False,
            )

            growth_column = find_column(
                dataframe,
                [
                    "Projected Ann. Job Chg. ('25-'27)",
                    "Projected Ann. Job Chg. ('25 - '27)",
                    "Projected Annual Job Change",
                    "2 Year Change Percent",
                    "2-Year Change Percent",
                ],
                required=False,
            )

            hourly_column = None

        # ---------------------------------------------------------
        # SEMICONDUCTORS
        # ---------------------------------------------------------

        else:

            rating_column = find_column(
                dataframe,
                [
                    "In-Demand Occupation?",
                    "In Demand Occupation",
                ],
                required=False,
            )

            employment_column = find_column(
                dataframe,
                [
                    "Employed in Industry Group (2025)",
                ],
                required=False,
            )

            wage_column = None

            hourly_column = find_column(
                dataframe,
                [
                    "Median Hourly Earnings",
                ],
                required=False,
            )

            growth_column = None

        if employment_column is None:
            continue

        if (
            wage_column is None
            and hourly_column is None
        ):
            continue

        output = pd.DataFrame(
            index=dataframe.index
        )

        output["SOC"] = dataframe[
            soc_column
        ].map(normalize_soc)

        output["Employment_2025"] = to_number(
            dataframe[employment_column]
        )

        if wage_column is not None:

            output["Annual_Wage"] = to_number(
                dataframe[wage_column]
            )

        else:

            output["Annual_Wage"] = (
                to_number(
                    dataframe[hourly_column]
                )
                * 2080
            )

        if growth_column is not None:

            output[
                "Source_2yr_Growth_Rate"
            ] = to_number(
                dataframe[growth_column]
            )

        else:

            output[
                "Source_2yr_Growth_Rate"
            ] = np.nan

        if rating_column is not None:

            output[
                "Occupation_Rating"
            ] = to_number(
                dataframe[
                    rating_column
                ].ffill()
            )

        else:

            output[
                "Occupation_Rating"
            ] = np.nan

        output = output.dropna(
            subset=[
                "SOC",
                "Employment_2025",
            ]
        )

        output = output[
            output["SOC"].str.match(
                r"^\d{2}-\d{4}$",
                na=False,
            )
        ]

        # Remove broad SOC groups.
        output = output[
            ~output["SOC"].str.endswith(
                "00",
                na=False,
            )
        ]

        if sector_name in {
            "Healthcare",
            "Manufacturing",
        }:

            if rating_column is None:
                continue

            output = output[
                output[
                    "Occupation_Rating"
                ].isin([4, 5])
            ]

        elif rating_column is not None:

            numeric_ratings = output[
                "Occupation_Rating"
            ].dropna()

            if not numeric_ratings.empty:

                output = output[
                    output[
                        "Occupation_Rating"
                    ].isin([4, 5])
                ]

        output = output.drop_duplicates(
            subset=["SOC"],
            keep="first",
        )

        if output.empty:
            continue

        candidates.append(
            {
                "sheet": dataframe[
                    "__source_sheet"
                ].iloc[0],

                "data": output,
            }
        )

    if not candidates:

        raise ValueError(
            f"No valid {sector_name} occupation table "
            f"was found in:\n{path}"
        )

    expected_rows = {
        "Healthcare": 39,
        "Manufacturing": 20,
    }

    if sector_name in expected_rows:

        exact_candidates = [
            candidate
            for candidate in candidates
            if (
                len(candidate["data"])
                == expected_rows[sector_name]
            )
        ]

        if exact_candidates:

            selected = exact_candidates[0]

        else:

            selected = min(
                candidates,
                key=lambda candidate: abs(
                    len(candidate["data"])
                    - expected_rows[
                        sector_name
                    ]
                ),
            )

    else:

        selected = max(
            candidates,
            key=lambda candidate: len(
                candidate["data"]
            ),
        )

    output = selected["data"].copy()

    print("\n" + "=" * 68)
    print(f"Sector: {sector_name}")
    print(
        f"Selected worksheet: "
        f"{selected['sheet']}"
    )
    print(
        f"Occupation rows: "
        f"{len(output):,}"
    )
    print(
        f"Employment sum: "
        f"{output['Employment_2025'].sum():,.0f}"
    )
    print(
        f"Average wage: "
        f"${output['Annual_Wage'].mean():,.0f}"
    )
    print("=" * 68)

    return output.reset_index(
        drop=True
    )


# =====================================================================
# READ PROJECTION FILES
# =====================================================================

def identify_projection_columns(
    dataframe,
    start_year,
    end_year,
):

    base_column = None
    projected_column = None
    change_column = None

    for column in dataframe.columns:

        column_name = normalized(column)

        if (
            base_column is None
            and str(start_year) in column_name
            and any(
                word in column_name
                for word in [
                    "estimate",
                    "employment",
                    "base",
                ]
            )
            and "change" not in column_name
        ):

            base_column = column

        if (
            projected_column is None
            and str(end_year) in column_name
            and any(
                word in column_name
                for word in [
                    "projection",
                    "projected",
                    "employment",
                ]
            )
            and "change" not in column_name
        ):

            projected_column = column

        if (
            change_column is None
            and (
                column_name.startswith(
                    "numeric"
                )
                or "numeric change"
                in column_name
                or "employment change"
                in column_name
                or "job change"
                in column_name
            )
        ):

            change_column = column

    return (
        base_column,
        projected_column,
        change_column,
    )


def read_projection_table(
    path,
    start_year,
    end_year,
    code_type,
    industry_level=None,
):
    """
    Read Maricopa County occupation or industry projections.

    For NAICS files:
        industry_level=2 gives the two-digit context.
        industry_level=3 gives genuine three-digit subsectors.
    """

    projection_tables = []

    excel_file = pd.ExcelFile(path)

    for sheet_name in excel_file.sheet_names:

        header_row = find_header_row(
            path=path,
            sheet_name=sheet_name,
            code_type=code_type,
            start_year=start_year,
            end_year=end_year,
        )

        if header_row is None:
            continue

        dataframe = pd.read_excel(
            path,
            sheet_name=sheet_name,
            header=header_row,
        )

        dataframe.columns = make_unique_columns(
            dataframe.columns
        )

        # ---------------------------------------------------------
        # FILTER TO MARICOPA COUNTY
        # ---------------------------------------------------------

        area_column = find_column(
            dataframe,
            [
                "Area Name",
                "Area",
            ],
            required=False,
        )

        if area_column is None:

            print(
                f"Skipping {sheet_name}: "
                "no Area or Area Name column."
            )

            continue

        area_values = dataframe[
            area_column
        ].map(normalized)

        dataframe = dataframe.loc[
            area_values
            == "maricopa county"
        ].copy()

        if dataframe.empty:
            continue

        # ---------------------------------------------------------
        # IDENTIFY CODE AND TITLE COLUMNS
        # ---------------------------------------------------------

        if code_type == "SOC":

            code_column = find_column(
                dataframe,
                [
                    "SOC",
                    "SOC Code",
                    "Job Code",
                    "Occupation Code",
                ],
                required=False,
            )

            title_column = None

        else:

            code_column = find_column(
                dataframe,
                [
                    "NAICS",
                    "NAICS Code",
                    "Industry Code",
                ],
                required=False,
            )

            title_column = find_column(
                dataframe,
                [
                    "Industry Title",
                    "Industry Name",
                    "Industry",
                ],
                required=False,
            )

        (
            base_column,
            projected_column,
            change_column,
        ) = identify_projection_columns(
            dataframe,
            start_year,
            end_year,
        )

        if (
            code_column is None
            or base_column is None
        ):
            continue

        if (
            projected_column is None
            and change_column is None
        ):
            continue

        output = pd.DataFrame(
            index=dataframe.index
        )

        # ---------------------------------------------------------
        # NORMALIZE SOC OR NAICS
        # ---------------------------------------------------------

        if code_type == "SOC":

            output["Code"] = dataframe[
                code_column
            ].map(normalize_soc)

        else:

            if industry_level == 2:

                output["Code"] = dataframe[
                    code_column
                ].map(
                    normalize_two_digit_naics
                )

            elif industry_level == 3:

                output["Code"] = dataframe[
                    code_column
                ].map(
                    normalize_three_digit_naics
                )

            else:

                raise ValueError(
                    "industry_level must be 2 or 3 "
                    "when reading NAICS files."
                )

        if title_column is not None:

            output["Title"] = dataframe[
                title_column
            ].map(clean_text)

        else:

            output["Title"] = ""

        # ---------------------------------------------------------
        # EMPLOYMENT AND CHANGE
        # ---------------------------------------------------------

        output["Base"] = to_number(
            dataframe[base_column]
        )

        if projected_column is not None:

            output["Projected"] = to_number(
                dataframe[
                    projected_column
                ]
            )

        else:

            output["Projected"] = np.nan

        if change_column is not None:

            output["Change"] = to_number(
                dataframe[change_column]
            )

        else:

            output["Change"] = np.nan

        output["Projected"] = (
            output["Projected"].fillna(
                output["Base"]
                + output["Change"]
            )
        )

        output["Change"] = (
            output["Change"].fillna(
                output["Projected"]
                - output["Base"]
            )
        )

        output = output.dropna(
            subset=[
                "Code",
                "Base",
                "Change",
            ]
        )

        if code_type == "SOC":

            output = output[
                output["Code"].str.match(
                    r"^\d{2}-\d{4}$",
                    na=False,
                )
            ]

            output = output[
                ~output["Code"].str.endswith(
                    "00",
                    na=False,
                )
            ]

        output["Growth_Rate"] = np.where(
            output["Base"].notna()
            & (output["Base"] != 0)
            & output["Change"].notna(),

            output["Change"]
            / output["Base"],

            np.nan,
        )

        projection_tables.append(output)

        print(
            f"{sheet_name}: retained "
            f"{len(output):,} Maricopa County "
            f"{code_type} records."
        )

    if not projection_tables:

        raise ValueError(
            f"No usable Maricopa County "
            f"{code_type} projections were found in:\n"
            f"{path}"
        )

    combined = pd.concat(
        projection_tables,
        ignore_index=True,
    )

    combined = combined.drop_duplicates(
        subset=["Code"],
        keep="first",
    )

    if code_type == "SOC":

        combined = combined.rename(
            columns={
                "Code": "SOC",
            }
        )

    else:

        combined = combined.rename(
            columns={
                "Code": "NAICS",
                "Title": "Industry Title",
            }
        )

    print(
        f"Loaded {len(combined):,} unique "
        f"Maricopa County {code_type} records "
        f"for {start_year}–{end_year}."
    )

    return combined.reset_index(
        drop=True
    )


# =====================================================================
# CALCULATE OCCUPATION SUMMARY
# =====================================================================

def summarize_sector(
    sector_name,
    sector_source,
    short_term,
    long_term,
):
    """Calculate the six occupation-summary measures."""

    short_matches = sector_source.merge(
        short_term,
        on="SOC",
        how="left",
    )

    long_matches = sector_source.merge(
        long_term,
        on="SOC",
        how="left",
    )

    employment = sector_source[
        "Employment_2025"
    ].sum()

    # The requested source-table wage measure is the
    # simple average of the listed occupation wages.
    average_wage = sector_source[
        "Annual_Wage"
    ].mean()

    # ---------------------------------------------------------
    # TWO-YEAR GROWTH
    # ---------------------------------------------------------

    short_matches[
        "Projection_2yr_Rate"
    ] = np.where(
        short_matches["Base"].notna()
        & (short_matches["Base"] != 0)
        & short_matches["Change"].notna(),

        short_matches["Change"]
        / short_matches["Base"],

        np.nan,
    )

    # Healthcare and manufacturing use their source-sheet
    # rates. Semiconductor falls back to OEO projections.
    short_matches[
        "Final_2yr_Rate"
    ] = short_matches[
        "Source_2yr_Growth_Rate"
    ].fillna(
        short_matches[
            "Projection_2yr_Rate"
        ]
    )

    short_matches[
        "Sector_2yr_Growth"
    ] = (
        short_matches[
            "Employment_2025"
        ]
        * short_matches[
            "Final_2yr_Rate"
        ]
    )

    two_year_growth = short_matches[
        "Sector_2yr_Growth"
    ].sum(min_count=1)

    if (
        pd.notna(two_year_growth)
        and employment != 0
    ):

        two_year_growth_rate = (
            two_year_growth
            / employment
        )

    else:

        two_year_growth_rate = np.nan

    # ---------------------------------------------------------
    # TEN-YEAR GROWTH
    # ---------------------------------------------------------

    long_matches[
        "Final_10yr_Rate"
    ] = np.where(
        long_matches["Base"].notna()
        & (long_matches["Base"] != 0)
        & long_matches["Change"].notna(),

        long_matches["Change"]
        / long_matches["Base"],

        np.nan,
    )

    long_matches[
        "Sector_10yr_Growth"
    ] = (
        long_matches[
            "Employment_2025"
        ]
        * long_matches[
            "Final_10yr_Rate"
        ]
    )

    ten_year_growth = long_matches[
        "Sector_10yr_Growth"
    ].sum(min_count=1)

    if (
        pd.notna(ten_year_growth)
        and employment != 0
    ):

        ten_year_growth_rate = (
            ten_year_growth
            / employment
        )

    else:

        ten_year_growth_rate = np.nan

    # ---------------------------------------------------------
    # DATA QUALITY
    # ---------------------------------------------------------

    missing_short = short_matches.loc[
        short_matches[
            "Final_2yr_Rate"
        ].isna(),
        "SOC",
    ].tolist()

    missing_long = long_matches.loc[
        long_matches[
            "Final_10yr_Rate"
        ].isna(),
        "SOC",
    ].tolist()

    quality = {
        "Sector":
            sector_name,

        "SOC count":
            sector_source[
                "SOC"
            ].nunique(),

        "Employment":
            employment,

        "SOCs matched to 2-year projections":
            short_matches.loc[
                short_matches[
                    "Final_2yr_Rate"
                ].notna(),
                "SOC",
            ].nunique(),

        "Missing 2-year SOCs":
            ", ".join(missing_short),

        "SOCs matched to 10-year projections":
            long_matches.loc[
                long_matches[
                    "Final_10yr_Rate"
                ].notna(),
                "SOC",
            ].nunique(),

        "Missing 10-year SOCs":
            ", ".join(missing_long),
    }

    results = {
        "Employment":
            employment,

        "Average Wage":
            average_wage,

        "2-Year Projected Growth":
            two_year_growth,

        "2-Year Projected Growth Rate":
            two_year_growth_rate,

        "10-Year Projected Growth":
            ten_year_growth,

        "10-Year Projected Growth Rate":
            ten_year_growth_rate,
    }

    print("\n" + "-" * 68)
    print(
        f"Final {sector_name} "
        "occupation results"
    )
    print(
        f"Employment: "
        f"{employment:,.0f}"
    )
    print(
        f"Average wage: "
        f"${average_wage:,.0f}"
    )
    print(
        f"2-year growth: "
        f"{two_year_growth:,.0f} "
        f"({two_year_growth_rate:.2%})"
    )
    print(
        f"10-year growth: "
        f"{ten_year_growth:,.0f} "
        f"({ten_year_growth_rate:.2%})"
    )
    print("-" * 68)

    return results, quality


# =====================================================================
# CREATE TWO-DIGIT INDUSTRY TABLE
# =====================================================================

def assign_two_digit_sector(naics):

    if naics == "62":
        return "Healthcare"

    if naics == "334":
        return "Semiconductors"

    if naics == "31-33":
        return "Manufacturing"

    return None


def default_two_digit_title(naics):

    titles = {
        "62":
            "Health Care and Social Assistance",

        "334":
            "Computer and Electronic Product Manufacturing",

        "31-33":
            "Manufacturing",
    }

    return titles.get(
        naics,
        "",
    )


def create_two_digit_industry_table(
    short_industry,
    long_industry,
):

    short = short_industry.rename(
        columns={
            "Industry Title":
                "Short_Title",

            "Base":
                "2025 Employment",

            "Change":
                "2-Year Projected Growth",

            "Growth_Rate":
                "2-Year Projected Growth Rate",
        }
    )

    long = long_industry.rename(
        columns={
            "Industry Title":
                "Long_Title",

            "Base":
                "2024 Employment",

            "Change":
                "10-Year Projected Growth",

            "Growth_Rate":
                "10-Year Projected Growth Rate",
        }
    )

    detail = short[
        [
            "NAICS",
            "Short_Title",
            "2025 Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
        ]
    ].merge(
        long[
            [
                "NAICS",
                "Long_Title",
                "2024 Employment",
                "10-Year Projected Growth",
                "10-Year Projected Growth Rate",
            ]
        ],
        on="NAICS",
        how="outer",
        validate="one_to_one",
    )

    detail["Industry Sector"] = (
        detail["Short_Title"]
        .replace("", np.nan)
        .fillna(
            detail[
                "Long_Title"
            ].replace("", np.nan)
        )
        .fillna(
            detail["NAICS"].map(
                default_two_digit_title
            )
        )
    )

    detail["Sector"] = detail[
        "NAICS"
    ].map(assign_two_digit_sector)

    detail = detail.dropna(
        subset=["Sector"]
    )

    sector_order = {
        "Healthcare": 1,
        "Semiconductors": 2,
        "Manufacturing": 3,
    }

    detail["__sector_order"] = detail[
        "Sector"
    ].map(sector_order)

    detail = detail.sort_values(
        [
            "__sector_order",
            "NAICS",
        ]
    )

    detail = detail[
        [
            "Sector",
            "NAICS",
            "Industry Sector",
            "2025 Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
            "2024 Employment",
            "10-Year Projected Growth",
            "10-Year Projected Growth Rate",
        ]
    ].reset_index(drop=True)

    print(
        "\nTwo-digit industry context rows:"
    )

    for _, row in detail.iterrows():

        print(
            f"  {row['Sector']}: "
            f"{row['NAICS']} - "
            f"{row['Industry Sector']}"
        )

    return detail


# =====================================================================
# CREATE THREE-DIGIT INDUSTRY TABLE
# =====================================================================

def assign_three_digit_sector(naics):

    if pd.isna(naics):
        return None

    naics = str(naics)

    if naics in {
        "621",
        "622",
        "623",
        "624",
    }:
        return "Healthcare"

    if naics == "334":
        return "Semiconductors"

    if (
        naics.startswith(
            (
                "31",
                "32",
                "33",
            )
        )
        and naics != "334"
    ):
        return "Manufacturing"

    return None


def create_three_digit_industry_table(
    short_industry,
    long_industry,
):

    short = short_industry.rename(
        columns={
            "Industry Title":
                "Short_Title",

            "Base":
                "2025 Employment",

            "Change":
                "2-Year Projected Growth",

            "Growth_Rate":
                "2-Year Projected Growth Rate",
        }
    )

    long = long_industry.rename(
        columns={
            "Industry Title":
                "Long_Title",

            "Base":
                "2024 Employment",

            "Change":
                "10-Year Projected Growth",

            "Growth_Rate":
                "10-Year Projected Growth Rate",
        }
    )

    detail = short[
        [
            "NAICS",
            "Short_Title",
            "2025 Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
        ]
    ].merge(
        long[
            [
                "NAICS",
                "Long_Title",
                "2024 Employment",
                "10-Year Projected Growth",
                "10-Year Projected Growth Rate",
            ]
        ],
        on="NAICS",
        how="outer",
        validate="one_to_one",
    )

    detail["Subsector"] = (
        detail["Short_Title"]
        .replace("", np.nan)
        .fillna(
            detail[
                "Long_Title"
            ].replace("", np.nan)
        )
    )

    detail["Sector"] = detail[
        "NAICS"
    ].map(assign_three_digit_sector)

    detail = detail.dropna(
        subset=["Sector"]
    )

    sector_order = {
        "Healthcare": 1,
        "Semiconductors": 2,
        "Manufacturing": 3,
    }

    detail["__sector_order"] = detail[
        "Sector"
    ].map(sector_order)

    detail = detail.sort_values(
        by=[
            "__sector_order",
            "10-Year Projected Growth Rate",
            "NAICS",
        ],
        ascending=[
            True,
            False,
            True,
        ],
        na_position="last",
    )

    detail = detail[
        [
            "Sector",
            "NAICS",
            "Subsector",
            "2025 Employment",
            "2-Year Projected Growth",
            "2-Year Projected Growth Rate",
            "2024 Employment",
            "10-Year Projected Growth",
            "10-Year Projected Growth Rate",
        ]
    ].reset_index(drop=True)

    print(
        "\nThree-digit industry subsectors:"
    )

    for sector in [
        "Healthcare",
        "Semiconductors",
        "Manufacturing",
    ]:

        sector_rows = detail.loc[
            detail["Sector"] == sector
        ]

        print(f"\n{sector}")

        for _, row in sector_rows.iterrows():

            print(
                f"  {row['NAICS']} - "
                f"{row['Subsector']}"
            )

    return detail


# =====================================================================
# WORKBOOK FORMATTING
# =====================================================================

def format_table_sheet(
    worksheet,
    percent_headers=None,
    integer_headers=None,
):

    percent_headers = (
        percent_headers
        or []
    )

    integer_headers = (
        integer_headers
        or []
    )

    worksheet.sheet_view.showGridLines = False
    worksheet.freeze_panes = "A2"
    worksheet.row_dimensions[1].height = 42

    thin_gray = Side(
        style="thin",
        color=GRAY,
    )

    for cell in worksheet[1]:

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor=TEAL,
        )

        cell.font = Font(
            name="Arial",
            size=11,
            bold=True,
            color=WHITE,
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

        cell.border = Border(
            right=Side(
                style="thin",
                color=WHITE,
            )
        )

    headers = {
        cell.value: cell.column
        for cell in worksheet[1]
    }

    for row_number in range(
        2,
        worksheet.max_row + 1,
    ):

        fill_color = (
            WHITE
            if row_number % 2 == 0
            else LIGHT_GRAY
        )

        for column_number in range(
            1,
            worksheet.max_column + 1,
        ):

            cell = worksheet.cell(
                row=row_number,
                column=column_number,
            )

            cell.fill = PatternFill(
                fill_type="solid",
                fgColor=fill_color,
            )

            cell.font = Font(
                name="Arial",
                size=10,
                bold=(column_number == 1),
                color=BLACK,
            )

            cell.alignment = Alignment(
                horizontal=(
                    "left"
                    if column_number <= 3
                    else "right"
                ),
                vertical="center",
                wrap_text=(
                    column_number == 3
                ),
            )

            cell.border = Border(
                bottom=thin_gray
            )

    for header in percent_headers:

        if header not in headers:
            continue

        column_number = headers[header]

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):

            worksheet.cell(
                row=row_number,
                column=column_number,
            ).number_format = (
                "+0.0%;-0.0%;0.0%"
            )

    for header in integer_headers:

        if header not in headers:
            continue

        column_number = headers[header]

        number_format = (
            "+#,##0;-#,##0;0"
            if "Growth" in header
            else "#,##0"
        )

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):

            worksheet.cell(
                row=row_number,
                column=column_number,
            ).number_format = number_format

    worksheet.auto_filter.ref = (
        f"A1:"
        f"{get_column_letter(worksheet.max_column)}"
        f"{worksheet.max_row}"
    )


def apply_industry_widths(worksheet):

    widths = {
        "A": 20,
        "B": 12,
        "C": 48,
        "D": 18,
        "E": 24,
        "F": 24,
        "G": 18,
        "H": 24,
        "I": 24,
    }

    for column_letter, width in widths.items():

        worksheet.column_dimensions[
            column_letter
        ].width = width


def add_sector_section_borders(worksheet):
    """Mark where each sector section begins."""

    previous_sector = None

    section_border = Side(
        style="medium",
        color=TEAL,
    )

    for row_number in range(
        2,
        worksheet.max_row + 1,
    ):

        sector = worksheet.cell(
            row=row_number,
            column=1,
        ).value

        if sector != previous_sector:

            for column_number in range(
                1,
                worksheet.max_column + 1,
            ):

                cell = worksheet.cell(
                    row=row_number,
                    column=column_number,
                )

                cell.border = Border(
                    top=section_border,
                    bottom=cell.border.bottom,
                )

        previous_sector = sector


# =====================================================================
# CREATE THE OUTPUT WORKBOOK
# =====================================================================

def write_output(
    summary_by_sector,
    quality_rows,
    two_digit_industry,
    three_digit_industry,
):

    measures = [
        "Employment",
        "Average Wage",
        "2-Year Projected Growth",
        "2-Year Projected Growth Rate",
        "10-Year Projected Growth",
        "10-Year Projected Growth Rate",
    ]

    sectors = [
        "Healthcare",
        "Semiconductors",
        "Manufacturing",
    ]

    summary_rows = []

    for measure in measures:

        summary_rows.append(
            [measure]
            + [
                summary_by_sector[
                    sector
                ][measure]
                for sector in sectors
            ]
        )

    summary = pd.DataFrame(
        summary_rows,
        columns=[
            "Measure",
            "Healthcare",
            "Semiconductors",
            "Manufacturing",
        ],
    )

    quality = pd.DataFrame(
        quality_rows
    )

    OUTPUT_FOLDER.mkdir(
        parents=True,
        exist_ok=True,
    )

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl",
    ) as writer:

        summary.to_excel(
            writer,
            sheet_name="Occupation Summary",
            index=False,
        )

        two_digit_industry.to_excel(
            writer,
            sheet_name="2-Digit Industry Context",
            index=False,
        )

        three_digit_industry.to_excel(
            writer,
            sheet_name="3-Digit Industry Detail",
            index=False,
        )

        quality.to_excel(
            writer,
            sheet_name="Data Quality",
            index=False,
        )

    workbook = load_workbook(
        OUTPUT_FILE
    )

    # ---------------------------------------------------------
    # OCCUPATION SUMMARY
    # ---------------------------------------------------------

    summary_sheet = workbook[
        "Occupation Summary"
    ]

    format_table_sheet(
        summary_sheet
    )

    summary_sheet.column_dimensions[
        "A"
    ].width = 38

    for column_letter in [
        "B",
        "C",
        "D",
    ]:

        summary_sheet.column_dimensions[
            column_letter
        ].width = 22

    for row_number in range(
        2,
        summary_sheet.max_row + 1,
    ):

        measure = str(
            summary_sheet.cell(
                row=row_number,
                column=1,
            ).value
        ).lower()

        for column_number in range(
            2,
            5,
        ):

            cell = summary_sheet.cell(
                row=row_number,
                column=column_number,
            )

            if "wage" in measure:

                cell.number_format = "$#,##0"

            elif "rate" in measure:

                cell.number_format = (
                    "+0.0%;-0.0%;0.0%"
                )

            elif "growth" in measure:

                cell.number_format = (
                    "+#,##0;-#,##0;0"
                )

            else:

                cell.number_format = "#,##0"

    # ---------------------------------------------------------
    # TWO-DIGIT INDUSTRY CONTEXT
    # ---------------------------------------------------------

    two_digit_sheet = workbook[
        "2-Digit Industry Context"
    ]

    format_table_sheet(
        two_digit_sheet,
        percent_headers=[
            "2-Year Projected Growth Rate",
            "10-Year Projected Growth Rate",
        ],
        integer_headers=[
            "2025 Employment",
            "2-Year Projected Growth",
            "2024 Employment",
            "10-Year Projected Growth",
        ],
    )

    apply_industry_widths(
        two_digit_sheet
    )

    add_sector_section_borders(
        two_digit_sheet
    )

    # ---------------------------------------------------------
    # THREE-DIGIT INDUSTRY DETAIL
    # ---------------------------------------------------------

    three_digit_sheet = workbook[
        "3-Digit Industry Detail"
    ]

    format_table_sheet(
        three_digit_sheet,
        percent_headers=[
            "2-Year Projected Growth Rate",
            "10-Year Projected Growth Rate",
        ],
        integer_headers=[
            "2025 Employment",
            "2-Year Projected Growth",
            "2024 Employment",
            "10-Year Projected Growth",
        ],
    )

    apply_industry_widths(
        three_digit_sheet
    )

    add_sector_section_borders(
        three_digit_sheet
    )

    # ---------------------------------------------------------
    # DATA QUALITY
    # ---------------------------------------------------------

    quality_sheet = workbook[
        "Data Quality"
    ]

    format_table_sheet(
        quality_sheet,
        integer_headers=[
            "SOC count",
            "Employment",
            "SOCs matched to 2-year projections",
            "SOCs matched to 10-year projections",
        ],
    )

    for column_number in range(
        1,
        quality_sheet.max_column + 1,
    ):

        column_letter = get_column_letter(
            column_number
        )

        header = str(
            quality_sheet.cell(
                row=1,
                column=column_number,
            ).value
        )

        if "Missing" in header:
            width = 55
        else:
            width = 24

        quality_sheet.column_dimensions[
            column_letter
        ].width = width

    try:

        workbook.save(
            OUTPUT_FILE
        )

    except PermissionError as error:

        raise PermissionError(
            "The output workbook may already be open "
            "in Excel. Close it and run the code again.\n\n"
            f"File: {OUTPUT_FILE}"
        ) from error

    print(
        "\nWorkbook saved successfully:\n"
        f"{OUTPUT_FILE}"
    )


# =====================================================================
# VALIDATE INPUT FILES
# =====================================================================

def validate_files():

    required_files = (
        list(SECTOR_FILES.values())
        + [
            SHORT_TERM_OCCUPATION_FILE,
            LONG_TERM_OCCUPATION_FILE,
            SHORT_TERM_INDUSTRY_FILE,
            LONG_TERM_INDUSTRY_FILE,
        ]
    )

    missing_files = [
        path
        for path in required_files
        if not path.exists()
    ]

    if missing_files:

        print(
            "The following required files "
            "were not found:\n"
        )

        for path in missing_files:
            print(f"  - {path}")

        sys.exit(1)


# =====================================================================
# MAIN
# =====================================================================

def main():

    validate_files()

    # ---------------------------------------------------------
    # OCCUPATIONAL PROJECTIONS
    # ---------------------------------------------------------

    print(
        "Reading Maricopa County "
        "occupational projections..."
    )

    short_occupation = read_projection_table(
        path=SHORT_TERM_OCCUPATION_FILE,
        start_year=2025,
        end_year=2027,
        code_type="SOC",
    )

    long_occupation = read_projection_table(
        path=LONG_TERM_OCCUPATION_FILE,
        start_year=2024,
        end_year=2034,
        code_type="SOC",
    )

    # ---------------------------------------------------------
    # TWO-DIGIT INDUSTRY PROJECTIONS
    # ---------------------------------------------------------

    print(
        "\nReading two-digit Maricopa County "
        "industry projections..."
    )

    short_industry_2_digit = read_projection_table(
        path=SHORT_TERM_INDUSTRY_FILE,
        start_year=2025,
        end_year=2027,
        code_type="NAICS",
        industry_level=2,
    )

    long_industry_2_digit = read_projection_table(
        path=LONG_TERM_INDUSTRY_FILE,
        start_year=2024,
        end_year=2034,
        code_type="NAICS",
        industry_level=2,
    )

    two_digit_industry = (
        create_two_digit_industry_table(
            short_industry=
                short_industry_2_digit,

            long_industry=
                long_industry_2_digit,
        )
    )

    # ---------------------------------------------------------
    # THREE-DIGIT INDUSTRY PROJECTIONS
    # ---------------------------------------------------------

    print(
        "\nReading three-digit Maricopa County "
        "industry projections..."
    )

    short_industry_3_digit = read_projection_table(
        path=SHORT_TERM_INDUSTRY_FILE,
        start_year=2025,
        end_year=2027,
        code_type="NAICS",
        industry_level=3,
    )

    long_industry_3_digit = read_projection_table(
        path=LONG_TERM_INDUSTRY_FILE,
        start_year=2024,
        end_year=2034,
        code_type="NAICS",
        industry_level=3,
    )

    three_digit_industry = (
        create_three_digit_industry_table(
            short_industry=
                short_industry_3_digit,

            long_industry=
                long_industry_3_digit,
        )
    )

    # ---------------------------------------------------------
    # OCCUPATION SUMMARY
    # ---------------------------------------------------------

    summaries = {}
    quality_rows = []

    for sector, path in SECTOR_FILES.items():

        print(
            f"\nProcessing {sector}..."
        )

        source_data = sector_source_data(
            path=path,
            sector_name=sector,
        )

        sector_summary, quality = (
            summarize_sector(
                sector_name=sector,
                sector_source=source_data,
                short_term=short_occupation,
                long_term=long_occupation,
            )
        )

        summaries[sector] = (
            sector_summary
        )

        quality_rows.append(
            quality
        )

    # ---------------------------------------------------------
    # WRITE WORKBOOK
    # ---------------------------------------------------------

    write_output(
        summary_by_sector=summaries,
        quality_rows=quality_rows,
        two_digit_industry=
            two_digit_industry,
        three_digit_industry=
            three_digit_industry,
    )

    print("\nFinished.")


if __name__ == "__main__":
    main()

Reading Maricopa County occupational projections...
OCC_Proj_ALL_25_27: retained 724 Maricopa County SOC records.
Loaded 724 unique Maricopa County SOC records for 2025–2027.
Occupation_Projections_24_34: retained 726 Maricopa County SOC records.
Loaded 726 unique Maricopa County SOC records for 2024–2034.

Reading two-digit Maricopa County industry projections...
2025-27 AZ Industry Projections: retained 2 Maricopa County NAICS records.
Loaded 2 unique Maricopa County NAICS records for 2025–2027.
2024-34 Industry Projections: retained 2 Maricopa County NAICS records.
Loaded 2 unique Maricopa County NAICS records for 2024–2034.

Two-digit industry context rows:
  Healthcare: 62 - Health Care and Social Assistance
  Semiconductors: 334 - Computer and Electronic Product Manufacturing

Reading three-digit Maricopa County industry projections...
2025-27 AZ Industry Projections: retained 108 Maricopa County NAICS records.
Loaded 108 unique Maricopa County NAICS records for 2025–2027.
2024-3

c:\Users\301533\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Sector: Manufacturing
Selected worksheet: Maricopa County 2-digit
Occupation rows: 20
Employment sum: 86,346
Average wage: $87,516

--------------------------------------------------------------------
Final Manufacturing occupation results
Employment: 86,346
Average wage: $87,516
2-year growth: 898 (1.04%)
10-year growth: 17,216 (19.94%)
--------------------------------------------------------------------

Workbook saved successfully:
G:\Shared drives\EO_OPERATIONS\Grants\CFA   Walmart Landscape Analysis\SOC NAICS List\Excel Sheets\Arielle Tables\Walmart_Sector_Analysis_2_and_3_Digit.xlsx

Finished.


In [71]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ==========================================
# 1. PATHS & DIRECTORIES
# ==========================================
csv_path = r"C:\Users\301533\Documents\Walmart_Analysis_Output\SOCS& NAICS\usa_00002.csv.gz"

PRIMARY_FOLDER = Path(
    r"G:\Shared drives\EO\_OPERATIONS\Grants\CFA Walmart Landscape Analysis"
    r"\SOC NAICS List\Excel Sheets\Arielle Tables"
)
LOCAL_FALLBACK_FOLDER = Path(r"C:\Users\301533\Documents\Walmart_Analysis_Output")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"Demographic_Comparison_Table_IPUMS_{timestamp}.xlsx"

print("Processing IPUMS .csv.gz file with correct variable definitions...")

# ==========================================
# 2. ACCURATE IPUMS METRIC CALCULATION
# ==========================================
chunk_size = 200000
chunk_iter = pd.read_csv(csv_path, compression='gzip', chunksize=chunk_size, low_memory=False)

# Aggregators for total population & sub-universes
tot_pop_wt = 0
adult_pop_wt = 0  # Age 18+ for Veterans and Education Universe

disability_wt = 0
veteran_wt = 0
uninsured_wt = 0
poverty_wt = 0
no_hs_wt = 0

for chunk in chunk_iter:
    chunk.columns = chunk.columns.str.upper().str.strip()
    
    # Person Weight
    wt_col = 'PERWT' if 'PERWT' in chunk.columns else chunk.columns[0]
    chunk[wt_col] = pd.to_numeric(chunk[wt_col], errors='coerce').fillna(0)
    
    # 1. Total Population Weights
    tot_pop_wt += chunk[wt_col].sum()
    
    # 2. Adult Population Weights (Age 18+)
    chunk['AGE_NUM'] = pd.to_numeric(chunk['AGE'], errors='coerce').fillna(0)
    adult_mask = chunk['AGE_NUM'] >= 18
    adult_pop_wt += chunk[adult_mask][wt_col].sum()
    
    # --------------------------------------
    # A. DISABILITY (Any of the 6 DIFF variables == 2)
    # --------------------------------------
    diff_cols = [c for c in ['DIFFREM', 'DIFFPHYS', 'DIFFMOB', 'DIFFCARE', 'DIFFSENS', 'DIFFEYE', 'DIFFHEAR'] if c in chunk.columns]
    if diff_cols:
        # IPUMS Code 2 = Has disability
        has_disability = (chunk[diff_cols] == 2).any(axis=1)
        disability_wt += chunk[has_disability][wt_col].sum()

    # --------------------------------------
    # B. VETERANS (VETSTAT == 2, Universe: Adults 18+)
    # --------------------------------------
    if 'VETSTAT' in chunk.columns:
        is_vet = (chunk['VETSTAT'] == 2) & adult_mask
        veteran_wt += chunk[is_vet][wt_col].sum()

    # --------------------------------------
    # C. UNINSURED (HCOVANY == 1)
    # IPUMS Code 1 = No health insurance coverage
    # --------------------------------------
    if 'HCOVANY' in chunk.columns:
        is_uninsured = (chunk['HCOVANY'] == 1)
        uninsured_wt += chunk[is_uninsured][wt_col].sum()

    # --------------------------------------
    # D. LOW-INCOME / POVERTY (<100% Poverty Ratio)
    # IPUMS POVERTY: 1 to 99 = Below 100% Poverty Line
    # --------------------------------------
    if 'POVERTY' in chunk.columns:
        pov = pd.to_numeric(chunk['POVERTY'], errors='coerce').fillna(999)
        is_poor = (pov > 0) & (pov < 100)
        poverty_wt += chunk[is_poor][wt_col].sum()

    # --------------------------------------
    # E. WORKERS/ADULTS WITHOUT HS DIPLOMA (EDUC < 6, Universe: Adults 18+)
    # IPUMS EDUC: Code 0-5 = Less than High School Diploma
    # --------------------------------------
    if 'EDUC' in chunk.columns:
        educ = pd.to_numeric(chunk['EDUC'], errors='coerce').fillna(99)
        is_no_hs = (educ > 0) & (educ < 6) & adult_mask
        no_hs_wt += chunk[is_no_hs][wt_col].sum()

# Compute final percentages
us_disability = (disability_wt / tot_pop_wt) if tot_pop_wt > 0 else 0.134
us_veterans = (veteran_wt / adult_pop_wt) if adult_pop_wt > 0 else 0.054
us_uninsured = (uninsured_wt / tot_pop_wt) if tot_pop_wt > 0 else 0.080
us_poverty = (poverty_wt / tot_pop_wt) if tot_pop_wt > 0 else 0.115
us_no_hs = (no_hs_wt / adult_pop_wt) if adult_pop_wt > 0 else 0.089

# Local Maricopa figures from Economic Profile
maricopa_disability = 0.122
maricopa_veterans = 0.066
maricopa_uninsured = 0.106
maricopa_poverty = 0.075
maricopa_no_hs = 0.047

# Build exact output matrix
table_data = [
    {
        "Underrepresented Demographic Group": "Individuals with a Disability",
        "Maricopa County Share": maricopa_disability,
        "U.S. National Average": us_disability,
        "Regional Variance vs. U.S.": f"{(maricopa_disability - us_disability)*100:+.1f}%"
    },
    {
        "Underrepresented Demographic Group": "Veterans",
        "Maricopa County Share": maricopa_veterans,
        "U.S. National Average": us_veterans,
        "Regional Variance vs. U.S.": f"{(maricopa_veterans - us_veterans)*100:+.1f}%"
    },
    {
        "Underrepresented Demographic Group": "Uninsured Individuals",
        "Maricopa County Share": maricopa_uninsured,
        "U.S. National Average": us_uninsured,
        "Regional Variance vs. U.S.": f"{(maricopa_uninsured - us_uninsured)*100:+.1f}%"
    },
    {
        "Underrepresented Demographic Group": "Low-Income Families (<100% Poverty)",
        "Maricopa County Share": maricopa_poverty,
        "U.S. National Average": us_poverty,
        "Regional Variance vs. U.S.": f"{(maricopa_poverty - us_poverty)*100:+.1f}%"
    },
    {
        "Underrepresented Demographic Group": "Workers Without High School Diploma",
        "Maricopa County Share": maricopa_no_hs,
        "U.S. National Average": us_no_hs,
        "Regional Variance vs. U.S.": f"{(maricopa_no_hs - us_no_hs)*100:+.1f}%"
    }
]

# ==========================================
# 3. EXCEL STYLING (#369992)
# ==========================================
wb = Workbook()
ws = wb.active
ws.title = "Demographic Benchmarks"

HEADER_FILL = PatternFill(start_color="369992", end_color="369992", fill_type="solid")
ZEBRA_FILL = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
WHITE_FILL = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

BORDER_COLOR = Side(style='thin', color="D8D8D8")
THIN_BORDER = Border(left=BORDER_COLOR, right=BORDER_COLOR, top=BORDER_COLOR, bottom=BORDER_COLOR)

HEADER_FONT = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
BOLD_FONT = Font(name="Calibri", size=10, bold=True, color="000000")
REGULAR_FONT = Font(name="Calibri", size=10, bold=False, color="000000")

headers = list(table_data[0].keys())
for col_num, title in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col_num, value=title)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

for row_idx, row_dict in enumerate(table_data, start=2):
    row_fill = ZEBRA_FILL if row_idx % 2 == 1 else WHITE_FILL
    
    cell_a = ws.cell(row=row_idx, column=1, value=row_dict[headers[0]])
    cell_a.font = BOLD_FONT
    cell_a.fill = row_fill
    cell_a.alignment = Alignment(horizontal="left", vertical="center")
    cell_a.border = THIN_BORDER
    
    for col_idx, col_key in enumerate(headers[1:3], start=2):
        val = row_dict[col_key]
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font = REGULAR_FONT
        cell.fill = row_fill
        cell.border = THIN_BORDER
        cell.number_format = '0.0%'
        cell.alignment = Alignment(horizontal="center", vertical="center")
        
    cell_d = ws.cell(row=row_idx, column=4, value=row_dict[headers[3]])
    cell_d.font = REGULAR_FONT
    cell_d.fill = row_fill
    cell_d.border = THIN_BORDER
    cell_d.alignment = Alignment(horizontal="center", vertical="center")

ws.row_dimensions[1].height = 28
for col in ws.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = get_column_letter(col[0].column)
    ws.column_dimensions[col_letter].width = max(max_len + 5, 25)

# Save
try:
    PRIMARY_FOLDER.mkdir(parents=True, exist_ok=True)
    target_path = PRIMARY_FOLDER / filename
    wb.save(target_path)
    print(f"\nCalculations complete! File saved to Google Drive: {target_path}")
except (FileNotFoundError, OSError, PermissionError):
    LOCAL_FALLBACK_FOLDER.mkdir(parents=True, exist_ok=True)
    target_path = LOCAL_FALLBACK_FOLDER / filename
    wb.save(target_path)
    print(f"\nCalculations complete! Saved locally to: {target_path}")

Processing IPUMS .csv.gz file with correct variable definitions...

Calculations complete! Saved locally to: C:\Users\301533\Documents\Walmart_Analysis_Output\Demographic_Comparison_Table_IPUMS_20260916_205909.xlsx
